# Field validation — `frontogenesis` (SURF pipeline)

End-to-end validation of every calculated field in the **frontogenesis** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `frontogenesis` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Frontogenesis-function diagnostics: F(u,v) on the full flow, F(ug,vg) on the geostrophic flow (from Eta), their difference (ageostrophic), plus geostrophic velocities and the ∇b-modified Okubo-Weiss W*.  Shared intermediates: velocity Jacobian J (validated in kinematic.ipynb) and buoyancy gradient ∇b (|∇b|² = gradb2 is validated in frontal_structure.ipynb).

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "frontogenesis"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = (
    list(defn["model_data_feature_channels"])
    + list(defn.get("pipeline_model_channels", {}).get(PIPELINE, []))
    + list(defn["compute_features_channels"])
)

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `frontogenesis`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`):

computed — `frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo`, `Wstar`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| frontogenesis_tendency | s⁻² | F(u,v) from J and ∇b | J (U, V); ∇b (b ← σ₀ ← Theta, Salt) | `calculate_fields.frontogenesis_tendency` (`_frontogenesis_formula`) |
| ug | m s⁻¹ | −(g/f)·∂η/∂y | Eta, f, grid metrics | `calculate_fields.geostrophic_velocity` |
| vg | m s⁻¹ | +(g/f)·∂η/∂x | Eta, f, grid metrics | `calculate_fields.geostrophic_velocity` |
| frontogenesis_geo | s⁻² | F(ug,vg) | ug, vg gradients; ∇b | `calculate_fields.frontogenesis_geo` |
| frontogenesis_ageo | s⁻² | F(u,v) − F(ug,vg) | both tendencies | `surface_subsets.compute_frontogenesis` (inline) |
| Wstar | s⁻² | modified Okubo-Weiss W* = 4·sgn(l₂)·√(l₁²+l₂²) (Bachman 2021) | J, ∇b, f | `calculate_fields.modified_okubo_weiss` |

Raw/intermediates plotted in dependency columns: `Eta` (raw), rotated
`U`, `V`, and `buoyancy` (b), computed live below.  ∇b and J
components: tabled only (§2 convention — gradb2/kinematic finals span
them).

Processing operations: land masking; staggered→tracer interpolation
+ CS/SN rotation (U, V, J); native-grid differentiation (∇b, ∇η, J —
halo rim); f-division (ug, vg, W* — equatorial extremes expected);
face→lat-lon stitching; global downsampling.

### Raw inputs & intermediates computed live: Eta, U, V, b

The store holds only the six output channels; chains start from raw Eta, the rotated velocities, and buoyancy — computed here from the same OSN snapshot with the same code and stitched to the rect grid.

In [ ]:
# Live raw inputs + intermediates (same loaders as the pipeline).
import dbof.preprocessing.calculate_fields as calculate_fields
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

LIVE = ["Eta", "U", "V", "buoyancy"]

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta", "U", "V"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")

u_east, v_north = calculate_fields.geographic_velocity(
    ds_merge, xgrid)
ds_conv = ds_raw.assign({
    "Eta": ds_merge["Eta"],
    "U": u_east,
    "V": v_north,
    "buoyancy": calculate_fields.buoyancy_of_field(ds_merge),
})[LIVE]
mask = {"_land_mask": (ds_merge.hFacC == 0)}
live_chw = stitch_and_mask(ds_conv, LIVE, mask)   # (4, H, W)
print(f"live fields stitched: {LIVE}, shape {live_chw.shape}")

In [ ]:
# Slice every field to the validation domains; full-res arrays are
# released immediately after slicing to keep peak memory at ~1 array.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = {}
for k, ch in enumerate(LIVE):
    region_arrays[ch] = regions.select_all_regions(live_chw[k], XC, YC)
del live_chw
for ch in CHANNELS:
    arr = reader.get_channel_snapshot(ch)
    region_arrays[ch] = regions.select_all_regions(arr, XC, YC)
    del arr

for ch in region_arrays:
    x, y, sub = region_arrays[ch]["gulf_stream"]
    print(f"{ch:24s} gulf_stream {sub.shape}  "
          f"min {np.nanmin(sub):.3g}  max {np.nanmax(sub):.3g}")

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → intermediate → final), rows = validation domains.  One
  shared colour scale per column; land/halo NaNs gray; regional
  boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Figure 3 — literature comparison**: (a) our generated data,
  (b) `.png` from `../literature_figures/` (named
  `{field}_{Citation}_{description}.png`); figure type specified per
  variable (`global_view=True` when the reference is global).
  Placeholder until a reference is supplied.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2).
CHAINS = {
    "frontogenesis_tendency": ["U", "V", "buoyancy", "frontogenesis_tendency"],
    "ug": ["Eta", "ug"],
    "vg": ["Eta", "vg"],
    "frontogenesis_geo": ["ug", "vg", "buoyancy", "frontogenesis_geo"],
    "frontogenesis_ageo": ["frontogenesis_tendency", "frontogenesis_geo", "frontogenesis_ageo"],
    "Wstar": ["U", "V", "buoyancy", "Wstar"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = {"Wstar", "frontogenesis_ageo", "frontogenesis_geo", "ug", "vg"}

# Fields drawn/binned on log scales (none in this subset unless set).
LOG_FIELDS = set()

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains")


def _pdf_arrays(field):
    """Region arrays for the PDF grid of one field.

    Applies the |lat|>2 deg filter to the Eq. Pacific row for
    f-normalised chain members (filter stated in the figure title).
    Inputs: field (str).  Outputs: dict like region_arrays.
    Generated by LH and Claude
    """
    out = {}
    for f in CHAINS[field]:
        d = dict(region_arrays[f])
        if f in F_NORM:
            x, y, a = d["eq_pacific"]
            d["eq_pacific"] = (x, y,
                               np.where(np.abs(y) > 2.0, a, np.nan))
        out[f] = d
    return out


def figure1_maps(field):
    """Figure 1: dependency-chain map grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | f-normalised: Eq. Pacific extreme near equator "
            "(expected)" if field in F_NORM else "")
    pipeline_map_grid(
        CHAINS[field], region_arrays, CMAP_CFG,
        diverging_cmaps=DIVERGING, log_scale_channels=LOG_FIELDS,
        suptitle=f"Figure 1 \u2014 {field}: pipeline maps{note}",
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: dependency-chain PDF grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | Eq. Pacific: |lat|>2\u00b0 filter (f-normalised)"
            if set(CHAINS[field]) & F_NORM else "")
    pipeline_pdf_grid(
        CHAINS[field], _pdf_arrays(field), CMAP_CFG,
        log10_fields=LOG_FIELDS,
        suptitle=f"Figure 2 \u2014 {field}: {PDF_NOTE}{note}",
    )
    plt.show()


def figure3_literature(field, png_name=None, caption=None,
                       global_view=False):
    """Figure 3: our data vs literature .png for one field.

    Inputs: field (str); png_name (str or None) — file in LIT_DIR;
    caption (str or None) — discussion text; global_view (bool) —
    render OUR panel as a global Robinson map (use when the
    literature figure is a global view) instead of the default
    Gulf Stream regional map.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    region = "global" if global_view else "gulf_stream"

    def _render(ax):
        x, y, arr = region_arrays[field][region]
        if global_view:
            # Same seam/Arctic handling as the Figure 1 global row.
            arr = mask_wrap_cells(x, y, arr)
        ax.set_facecolor(LAND_COLOR)
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            log_scale_channels=LOG_FIELDS, diverging_cmaps=DIVERGING,
            transform=ccrs.PlateCarree() if global_view else None,
            add_coastline=global_view,
            coastline_kw={"linewidth": 0.4, "edgecolor": "k"},
        )
        if im is not None:
            plt.colorbar(im, ax=ax, orientation="horizontal",
                         fraction=0.04, pad=0.04, label=label)

    side_by_side(
        _render, LIT_DIR / png_name if png_name else None,
        projection=ccrs.Robinson() if global_view else None,
        caption=caption or ("Discussion: awaiting literature "
                            f"reference for {field}."),
    )
    plt.show()

### 5.1 frontogenesis_tendency

F(u,v): rate of frontal sharpening by the full flow.  Positive filaments along strain-dominated fronts (Gulf Stream north wall, ACC).

In [ ]:
figure1_maps("frontogenesis_tendency")

In [ ]:
figure2_pdfs("frontogenesis_tendency")

In [ ]:
figure3_literature("frontogenesis_tendency")

### 5.2 ug (geostrophic U)

−(g/f)η_y.  Should track the rotated U at mesoscale away from the equator; equatorial blow-up expected (f→0).

In [ ]:
figure1_maps("ug")

In [ ]:
figure2_pdfs("ug")

In [ ]:
figure3_literature("ug")

### 5.3 vg (geostrophic V)

+(g/f)η_x.  Same checks as ug, meridional component.

In [ ]:
figure1_maps("vg")

In [ ]:
figure2_pdfs("vg")

In [ ]:
figure3_literature("vg")

### 5.4 frontogenesis_geo

F(ug,vg): geostrophic contribution to frontal sharpening; smoother than the full tendency.

In [ ]:
figure1_maps("frontogenesis_geo")

In [ ]:
figure2_pdfs("frontogenesis_geo")

In [ ]:
figure3_literature("frontogenesis_geo")

### 5.5 frontogenesis_ageo

F(u,v) − F(ug,vg): ageostrophic/submesoscale contribution; dispatcher-inline difference (see table).

In [ ]:
figure1_maps("frontogenesis_ageo")

In [ ]:
figure2_pdfs("frontogenesis_ageo")

In [ ]:
figure3_literature("frontogenesis_ageo")

### 5.6 Wstar (modified Okubo-Weiss)

Bachman (2021) W*: Q-vector-sensitive eddy/strain partition using ∇b and f; NaN-heavy at the equator (expected; PDF row filtered).

In [ ]:
figure1_maps("Wstar")

In [ ]:
figure2_pdfs("Wstar")

In [ ]:
figure3_literature("Wstar")

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

**Cross-references** — J machinery → `kinematic.ipynb`; b and |∇b|² (gradb2) → `frontal_structure.ipynb`; raw Eta/U/V → `native_fields.ipynb`.